# NB13 — Nivel 2: EDA exploratorio y arquitectura del modelo personalizado RUNA/Strava

**Objetivo**: explorar los patrones de FC (frecuencia cardíaca) y ritmo en nuestros atletas reales
(datos de Strava vía Supabase) y diseñar la arquitectura **corregida** del Nivel 2 del modelo jerárquico.

**Contexto metodológico (tesis)**:
- Nivel 1: prior poblacional entrenado con Endomondo/FitRec (NB12) → artefacto `nivel1_prior_poblacional_FULL_v5.pkl`
- Nivel 2 (**cohort-level generalizable**): sesiones propias de la cohorte RUNA + datos del formulario.
  **Sin CTL/ATL/ACWR** — esas variables longitudinales van al Nivel 3.
- Nivel 3 (**longitudinal individual**): historial acumulado del propio atleta (CTL/ATL/TSB/ACWR desde
  `weekly_features` Supabase + carreras propias). Sin check-ins.
- **Consentimiento**: todos los atletas firmaron el formulario de onboarding con consentimiento
  explícito para uso académico anonimizado (Ley 1581/2012, toggle `consent_anon`).
- **N elegibles Nivel 2**: **32 atletas** (≥8 semanas de historial + ≥10 runs con HR documentado;
  filtro `raw->>average_heartrate IS NOT NULL`, commit `83bf113`).

**Arquitectura vigente (2026-05-12 — THESIS_CONTEXT.md §5 y §8):**
```
N2 cohort-level: pred_nivel1 + age + sex_bin + vdot_estimated + avg_hr + pct_hrmax
                 + zona_hr + fcmax_obs + log_distance_km + elevation_gain_m
                 + avg_cadence + (day_of_week_sin/cos)
                 → sample_weight = 1/n_sesiones_atleta
                 → LOAO-CV 32 folds | naïveAutoML | Friedman-Nemenyi

N3 longitudinal: todo N2 + CTL + ATL + TSB + ACWR + own_race_best_pace + n_races
                 → activación: ≥3 carreras propias (sin check-ins)
```

**Estructura del notebook**:
- **Parte A — EDA exploratorio**: carga desde Supabase, perfiles, imbalance de sesiones,
  distribuciones HR/pace, auditoría de calidad HR, correlaciones, variabilidad inter-atleta,
  heatmap semanal, Ridge LOAO-CV baseline
- **Parte B — Arquitectura Nivel 2**: definición formal, feature set N2 (sin CTL/ATL),
  stacking pred_nivel1, sample weighting, naïveAutoML LOAO-CV, Friedman-Nemenyi,
  diagnóstico de errores, calibración conformal

## 0 · Setup y carga de entorno

In [ ]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# --- Paths ---
BASE = Path(r'C:/Users/andre/OneDrive/Documentos/Maestría Analítica Aplicada/running_coaching')
OUT_DIR = BASE / 'ml' / 'notebooks' / 'outputs' / 'nb13'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Agregar src/ al path para usar supabase_client
sys.path.insert(0, str(BASE))
load_dotenv(BASE / '.env')

# --- Estilo RUNA ---
CRIMSON = '#C41E3A'
NAVY    = '#1F4B99'
CREAM   = '#FDFBF7'
SLATE   = '#4B5563'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.prop_cycle': plt.cycler(color=[CRIMSON, NAVY, '#2563EB', '#059669', '#D97706', '#7C3AED']),
})

print('Setup OK — BASE:', BASE.exists())

## A · 1 — Carga de datos desde Supabase

In [ ]:
from src.storage.supabase_client import get_client

client = get_client()
assert client is not None, "Supabase no configurado — verifica .env"

# ── 1a. Perfiles de atletas ─────────────────────────────────────────────────
profiles_raw = (
    client.table("athlete_profiles")
    .select("cedula,age,sex,experience_years,weekly_km_current,main_sport,weekly_days")
    .execute()
).data or []

df_profiles = pd.DataFrame(profiles_raw)
print(f"athlete_profiles: {len(df_profiles)} atletas")
print(df_profiles.dtypes)
df_profiles.head()

In [ ]:
# ── 1b. Actividades (solo runs con HR) ─────────────────────────────────────
# La columna raw es JSONB — Supabase lo retorna ya como dict
acts_raw = (
    client.table("activities")
    .select("strava_id,cedula,sport_type,activity_date,distance_m,duration_sec,"
            "elevation_m,avg_pace_sec_km,raw")
    .eq("sport_type", "Run")
    .order("activity_date", desc=False)
    .execute()
).data or []

print(f"Actividades Run cargadas: {len(acts_raw)}")

rows = []
for r in acts_raw:
    raw = r.get("raw") or {}
    rows.append({
        "strava_id":       r["strava_id"],
        "cedula":          r["cedula"],
        "activity_date":   r["activity_date"],
        "distance_m":      r.get("distance_m"),
        "distance_km":     (r["distance_m"] / 1000.0) if r.get("distance_m") else None,
        "duration_sec":    r.get("duration_sec"),
        "duration_min":    (r["duration_sec"] / 60.0) if r.get("duration_sec") else None,
        "elevation_m":     r.get("elevation_m"),
        "pace_sec_km":     r.get("avg_pace_sec_km"),
        "pace_min_km":     (r["avg_pace_sec_km"] / 60.0) if r.get("avg_pace_sec_km") else None,
        "avg_hr":          raw.get("average_heartrate"),
        "max_hr":          raw.get("max_heartrate"),
        "avg_cadence":     raw.get("average_cadence"),
        "avg_speed_ms":    raw.get("average_speed"),
        "suffer_score":    raw.get("suffer_score"),
        "perceived_exertion": raw.get("perceived_exertion"),
    })

df_acts = pd.DataFrame(rows)
df_acts["activity_date"] = pd.to_datetime(df_acts["activity_date"], utc=True).dt.tz_localize(None)

print(f"\nShape: {df_acts.shape}")
print(f"Atletas únicos: {df_acts['cedula'].nunique()}")
print(f"\nCobertura HR (avg_hr no nulo): {df_acts['avg_hr'].notna().sum()} "
      f"({df_acts['avg_hr'].notna().mean()*100:.1f}%)")
df_acts.describe(percentiles=[.1,.25,.5,.75,.9]).round(1)

In [ ]:
# ── 1c. Weekly features desde Supabase ─────────────────────────────────────
wf_raw = (
    client.table("weekly_features")
    .select("cedula,week_start,total_km,total_runs,avg_pace_sec_km,avg_hr,"
            "ctl,atl,tsb,acwr,readiness_score")
    .order("week_start", desc=False)
    .execute()
).data or []

df_wf = pd.DataFrame(wf_raw)
if not df_wf.empty:
    df_wf["week_start"] = pd.to_datetime(df_wf["week_start"])
    numeric_cols = ["total_km","total_runs","avg_pace_sec_km","avg_hr","ctl","atl","tsb","acwr","readiness_score"]
    for c in numeric_cols:
        if c in df_wf.columns:
            df_wf[c] = pd.to_numeric(df_wf[c], errors="coerce")

print(f"weekly_features: {len(df_wf)} filas, {df_wf['cedula'].nunique() if not df_wf.empty else 0} atletas")
if not df_wf.empty:
    print(f"Rango temporal: {df_wf['week_start'].min().date()} → {df_wf['week_start'].max().date()}")
    print(df_wf.describe().round(2))

In [ ]:
# ── 1d. Merge actividades + perfiles → dataset maestro ─────────────────────
df = df_acts.merge(df_profiles, on="cedula", how="left")

# Filtros de coherencia
df = df[
    df["distance_km"].between(1.0, 60.0) &
    df["pace_min_km"].between(3.0, 12.0) &
    df["duration_min"].between(5.0, 300.0)
].copy()

# HR max estimado por edad (Tanaka: 208 - 0.7*age)
df["hr_max_est"] = np.where(df["age"].notna(), 208 - 0.7 * df["age"], np.nan)
df["pct_hr_max"] = np.where(
    df["avg_hr"].notna() & df["hr_max_est"].notna(),
    df["avg_hr"] / df["hr_max_est"],
    np.nan
)

# Zona HR (según % FCmax Karvonen clásico)
def zona_hr(pct):
    if pd.isna(pct): return np.nan
    if pct < 0.60: return 1
    if pct < 0.70: return 2
    if pct < 0.80: return 3
    if pct < 0.90: return 4
    return 5

df["zona_hr"] = df["pct_hr_max"].apply(zona_hr)

# Anonymize: usar índice numérico en lugar de cédula para figuras
cedula_to_idx = {c: i+1 for i, c in enumerate(sorted(df["cedula"].unique()))}
df["athlete_id"] = df["cedula"].map(cedula_to_idx)

print(f"Dataset maestro: {len(df)} actividades, {df['cedula'].nunique()} atletas")
print(f"\nActividades con HR: {df['avg_hr'].notna().sum()} ({df['avg_hr'].notna().mean()*100:.1f}%)")
print(f"Actividades con zona HR: {df['zona_hr'].notna().sum()}")
df[["pace_min_km","avg_hr","pct_hr_max","zona_hr","distance_km","age","sex"]].describe().round(2)

## A · 2 — Perfil demográfico del grupo

In [ ]:
# Resumen por atleta: cuántas actividades, HR coverage, rango temporal
per_athlete = (
    df.groupby("cedula").agg(
        n_runs=("strava_id", "count"),
        n_hr=("avg_hr", "count"),
        first_run=("activity_date", "min"),
        last_run=("activity_date", "max"),
        age=("age", "first"),
        sex=("sex", "first"),
        avg_pace_min_km=("pace_min_km", "mean"),
        avg_hr_mean=("avg_hr", "mean"),
        median_distance_km=("distance_km", "median"),
    ).reset_index()
)
per_athlete["weeks_span"] = (per_athlete["last_run"] - per_athlete["first_run"]).dt.days / 7
per_athlete["pct_hr"] = per_athlete["n_hr"] / per_athlete["n_runs"] * 100
per_athlete["athlete_id"] = per_athlete["cedula"].map(cedula_to_idx)

# ── Elegibles para Nivel 2 ──────────────────────────────────────────────────
WEEKS_MIN = 8
HR_RUNS_MIN = 10

eligible = per_athlete[
    (per_athlete["weeks_span"] >= WEEKS_MIN) &
    (per_athlete["n_hr"] >= HR_RUNS_MIN)
].copy()

print(f"Atletas totales: {len(per_athlete)}")
print(f"Elegibles Nivel 2 (≥{WEEKS_MIN} sem + ≥{HR_RUNS_MIN} runs con HR): {len(eligible)}")
print()
display_cols = ["athlete_id","sex","age","n_runs","n_hr","pct_hr","weeks_span","avg_pace_min_km","avg_hr_mean"]
per_athlete[display_cols].sort_values("n_runs", ascending=False).round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Perfil demográfico — Atletas RUNA', fontsize=13, fontweight='bold')

# Distribución de edad
ax = axes[0]
ax.hist(per_athlete["age"].dropna(), bins=10, color=CRIMSON, alpha=0.8, edgecolor='white')
ax.set_xlabel('Edad (años)')
ax.set_ylabel('N atletas')
ax.set_title('Distribución de edad')
ax.axvline(per_athlete["age"].median(), color=NAVY, lw=2, ls='--',
           label=f'Mediana {per_athlete["age"].median():.0f}a')
ax.legend(fontsize=9)

# Distribución por sexo
ax = axes[1]
sex_counts = per_athlete["sex"].value_counts()
ax.bar(sex_counts.index, sex_counts.values, color=[CRIMSON, NAVY], alpha=0.85)
for i, (label, val) in enumerate(sex_counts.items()):
    ax.text(i, val + 0.2, str(val), ha='center', fontweight='bold')
ax.set_title('Distribución por sexo')
ax.set_ylabel('N atletas')

# Actividades y cobertura HR por atleta
ax = axes[2]
pa_sorted = per_athlete.sort_values("n_runs", ascending=True)
ax.barh(pa_sorted["athlete_id"].astype(str), pa_sorted["n_runs"],
        alpha=0.35, color=NAVY, label="Runs totales")
ax.barh(pa_sorted["athlete_id"].astype(str), pa_sorted["n_hr"],
        alpha=0.85, color=CRIMSON, label="Runs con HR")
ax.set_xlabel("N actividades")
ax.set_title("Actividades por atleta")
ax.legend(fontsize=9)
ax.set_ylabel("ID atleta (anónimo)")

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig1_demografia.png', dpi=150, bbox_inches='tight')
plt.show()

## A · 2b — Imbalance de sesiones por atleta (motiva sample weighting en Parte B)

In [ ]:
# ── Distribución de sesiones por atleta ─────────────────────────────────────
# Muestra el imbalance severo que justifica sample_weight = 1/n_sesiones en Parte B

n_sessions = per_athlete.sort_values("n_runs", ascending=False).copy()
n_sessions["athlete_id"] = n_sessions["cedula"].map(cedula_to_idx)

print("DISTRIBUCIÓN DE SESIONES POR ATLETA")
print("=" * 50)
print(f"  Mínimo:     {n_sessions['n_runs'].min()}")
print(f"  Mediana:    {n_sessions['n_runs'].median():.0f}")
print(f"  Media:      {n_sessions['n_runs'].mean():.0f}")
print(f"  Máximo:     {n_sessions['n_runs'].max()}")
print(f"  Ratio max/min: {n_sessions['n_runs'].max() / n_sessions['n_runs'].min():.0f}x")

# Los 5 más grandes concentran qué % de las sesiones?
top5_sum = n_sessions.nlargest(5, "n_runs")["n_runs"].sum()
total_sum = n_sessions["n_runs"].sum()
print(f"\n  Top-5 atletas concentran {top5_sum/total_sum*100:.0f}% de todas las sesiones")
print(f"  → Sin sample_weight, el modelo optimizaría para esos 5 atletas")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Imbalance de sesiones — motiva sample_weight = 1/n_sesiones', fontsize=12, fontweight='bold')

# Panel 1: distribución lineal (muestra la cola larga)
ax = axes[0]
ax.barh(n_sessions["athlete_id"].astype(str), n_sessions["n_runs"],
        color=[CRIMSON if v > n_sessions["n_runs"].median() else NAVY for v in n_sessions["n_runs"]],
        alpha=0.8)
ax.axvline(n_sessions["n_runs"].median(), color='black', ls='--', lw=1.5,
           label=f'Mediana={n_sessions["n_runs"].median():.0f}')
ax.set_xlabel('N sesiones')
ax.set_ylabel('Atleta ID (anónimo)')
ax.set_title('Sesiones por atleta (escala lineal)\nRojo = sobre la mediana')
ax.legend(fontsize=9)

# Panel 2: histograma log-scale
ax = axes[1]
ax.hist(n_sessions["n_runs"], bins=20, color=NAVY, alpha=0.8, edgecolor='white', log=True)
ax.set_xlabel('N sesiones por atleta')
ax.set_ylabel('N atletas (escala log)')
ax.set_title('Distribución (escala log)\nConfirma distribución asimétrica')
ax.axvline(n_sessions["n_runs"].median(), color=CRIMSON, lw=2, ls='--',
           label=f'Mediana={n_sessions["n_runs"].median():.0f}')
ax.legend(fontsize=9)

# Tabla de sample weights para los 10 atletas con más sesiones
print("\nSample weights para top-10 atletas:")
print(f"{'ID':>5} | {'N sesiones':>12} | {'w = 1/n':>10} | {'N×w (contribución)':>20}")
print("-" * 55)
for _, row in n_sessions.head(10).iterrows():
    w = 1.0 / row["n_runs"]
    contribution = row["n_runs"] * w  # siempre 1.0
    print(f"{int(row['athlete_id']):>5} | {int(row['n_runs']):>12} | {w:>10.4f} | {contribution:>20.1f}")

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

## A · 3 — Distribuciones de ritmo y FC

In [ ]:
df_hr = df[df["avg_hr"].notna() & df["pace_min_km"].notna()].copy()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribuciones de ritmo y FC — Actividades RUNA', fontsize=13, fontweight='bold')

# 1. Histograma ritmo global
ax = axes[0, 0]
ax.hist(df["pace_min_km"].dropna(), bins=40, color=CRIMSON, alpha=0.8, edgecolor='white')
ax.set_xlabel('Ritmo (min/km)')
ax.set_title('Distribución global de ritmo')
ax.axvline(df["pace_min_km"].median(), color=NAVY, lw=2, ls='--',
           label=f'Mediana: {df["pace_min_km"].median():.1f} min/km')
ax.legend(fontsize=9)

# 2. Histograma FC media
ax = axes[0, 1]
ax.hist(df_hr["avg_hr"], bins=40, color=NAVY, alpha=0.8, edgecolor='white')
ax.set_xlabel('FC media (bpm)')
ax.set_title('Distribución global de FC media')
ax.axvline(df_hr["avg_hr"].median(), color=CRIMSON, lw=2, ls='--',
           label=f'Mediana: {df_hr["avg_hr"].median():.0f} bpm')
ax.legend(fontsize=9)

# 3. % FCmax
ax = axes[0, 2]
pct_vals = df_hr["pct_hr_max"].dropna() * 100
ax.hist(pct_vals, bins=40, color='#059669', alpha=0.8, edgecolor='white')
ax.set_xlabel('% FCmax estimada (Tanaka)')
ax.set_title('Intensidad relativa de sesiones')
for z, label, color in [(60,'Z1','#93C5FD'), (70,'Z2','#6EE7B7'), (80,'Z3','#FCD34D'),
                         (90,'Z4','#F87171'), (100,'Z5','#C41E3A')]:
    ax.axvline(z, color=color, lw=1.5, alpha=0.7)
ax.text(62, ax.get_ylim()[1]*0.9, 'Z1-Z5', fontsize=8, color=SLATE)

# 4. Boxplot ritmo por sexo
ax = axes[1, 0]
df_sex = df[df["sex"].notna()]
sexes = df_sex["sex"].unique()
data_by_sex = [df_sex[df_sex["sex"] == s]["pace_min_km"].dropna().values for s in sexes]
bp = ax.boxplot(data_by_sex, patch_artist=True,
                boxprops=dict(facecolor=CRIMSON, alpha=0.6),
                medianprops=dict(color=NAVY, lw=2))
ax.set_xticks(range(1, len(sexes)+1))
ax.set_xticklabels(sexes)
ax.set_ylabel('Ritmo (min/km)')
ax.set_title('Ritmo por sexo')

# 5. Boxplot FC por sexo
ax = axes[1, 1]
data_hr_by_sex = [df_sex[df_sex["sex"] == s]["avg_hr"].dropna().values for s in sexes]
ax.boxplot(data_hr_by_sex, patch_artist=True,
           boxprops=dict(facecolor=NAVY, alpha=0.6),
           medianprops=dict(color=CRIMSON, lw=2))
ax.set_xticks(range(1, len(sexes)+1))
ax.set_xticklabels(sexes)
ax.set_ylabel('FC media (bpm)')
ax.set_title('FC media por sexo')

# 6. Distribución zonas HR
ax = axes[1, 2]
zona_counts = df_hr["zona_hr"].dropna().value_counts().sort_index()
zone_colors = ['#93C5FD', '#6EE7B7', '#FCD34D', '#F87171', '#C41E3A']
bars = ax.bar([f'Z{int(z)}' for z in zona_counts.index], zona_counts.values,
              color=zone_colors[:len(zona_counts)], alpha=0.85)
ax.set_xlabel('Zona HR (% FCmax Karvonen)')
ax.set_ylabel('N actividades')
ax.set_title('Distribución de zonas de intensidad')
for bar, val in zip(bars, zona_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val}\n({val/len(df_hr)*100:.0f}%)', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig2_distribuciones.png', dpi=150, bbox_inches='tight')
plt.show()

## A · 3b — Auditoría de calidad HR (diagnóstico por atleta)

**Objetivo**: identificar sesiones con datos de FC sospechosos o imposibles *antes* de modelar.
La estrategia es **marcar primero (flag), eliminar solo lo físicamente imposible,
y windsorizar lo sospechoso** — preservando el máximo de datos posible dado el N pequeño.

| Flag | Criterio | Acción |
|------|----------|--------|
| `flag_low_hr` | avg_hr < 100 bpm | ⚠️ sospechoso — windsorizar |
| `flag_high_hr` | avg_hr > 195 bpm | ⚠️ sospechoso (ruido sensor muñeca) — windsorizar |
| `flag_low_pct` | %FCmax < 45% | ⚠️ sospechoso — revisar FCmax_obs del atleta |
| `flag_high_pct` | %FCmax > 105% | ⚠️ FCmax_obs subestimada — windsorizar |
| `flag_flat` | max_hr − avg_hr < 5 bpm | ⚠️ posible pérdida de contacto del sensor |
| `flag_impossible` | avg_hr < 50 o > 220 bpm | ❌ **eliminar** — fisiológicamente imposible |

> **"Diagnóstico por atleta"**: tabla resumen con quality tier por atleta (🟢🟡🟠🔴).
> Permite identificar quiénes tienen configuración Strava problemática (privacidad HR,
> sensor de muñeca sin calibrar) para contactarlos antes del entrenamiento del Nivel 2.

In [ ]:
# ── Umbrales de calidad HR ───────────────────────────────────────────────────
HR_MIN_RUN  = 100    # bpm — mínimo sospechoso en running sostenido
HR_MAX_RUN  = 195    # bpm — máximo sospechoso (ruido sensor muñeca)
PCT_MIN     = 0.45   # % FCmax mínimo razonable en carrera (Z1 ≈ 60%)
PCT_MAX     = 1.05   # % FCmax máximo (>100% = FCmax_obs subestimada)
FLAT_DELTA  = 5      # bpm — diferencia max_hr − avg_hr mínima esperada
HR_IMP_LOW  = 50     # bpm — imposible fisiológico
HR_IMP_HIGH = 220    # bpm — imposible fisiológico

df_all = df.copy()   # conservar df original sin modificar

# ── Aplicar flags (solo sobre sesiones con avg_hr disponible) ────────────────
has_hr = df_all["avg_hr"].notna()
df_all["flag_low_hr"]     = has_hr & (df_all["avg_hr"] < HR_MIN_RUN)
df_all["flag_high_hr"]    = has_hr & (df_all["avg_hr"] > HR_MAX_RUN)
df_all["flag_low_pct"]    = df_all["pct_hr_max"].notna() & (df_all["pct_hr_max"] < PCT_MIN)
df_all["flag_high_pct"]   = df_all["pct_hr_max"].notna() & (df_all["pct_hr_max"] > PCT_MAX)
df_all["flag_flat"]       = (
    has_hr & df_all["max_hr"].notna() &
    ((df_all["max_hr"] - df_all["avg_hr"]) < FLAT_DELTA)
)
df_all["flag_impossible"] = has_hr & (
    (df_all["avg_hr"] < HR_IMP_LOW) | (df_all["avg_hr"] > HR_IMP_HIGH)
)

FLAG_COLS = ["flag_low_hr", "flag_high_hr", "flag_low_pct",
             "flag_high_pct", "flag_flat", "flag_impossible"]
FLAG_LABELS = {
    "flag_low_hr":    f"HR baja (<{HR_MIN_RUN} bpm)",
    "flag_high_hr":   f"HR alta (>{HR_MAX_RUN} bpm)",
    "flag_low_pct":   f"%FCmax baja (<{int(PCT_MIN*100)}%)",
    "flag_high_pct":  f"%FCmax alta (>{int(PCT_MAX*100)}%)",
    "flag_flat":      "Sensor plano (max-avg < 5 bpm)",
    "flag_impossible":"HR imposible (<50 o >220 bpm)",
}

df_all["any_flag"] = df_all[FLAG_COLS].any(axis=1)
df_all["n_flags"]  = df_all[FLAG_COLS].sum(axis=1)

# ── Auditoría global ─────────────────────────────────────────────────────────
n_with_hr = int(df_all["avg_hr"].notna().sum())
print("=" * 62)
print("AUDITORIA GLOBAL DE CALIDAD HR")
print("=" * 62)
print(f"Total sesiones run (pace valido): {len(df_all)}")
print(f"  con avg_hr registrado:          {n_with_hr} ({n_with_hr/len(df_all)*100:.1f}%)")
print()
for col, label in FLAG_LABELS.items():
    n   = int(df_all[col].sum())
    pct = n / n_with_hr * 100 if n_with_hr > 0 else 0.0
    print(f"  {label:44s}: {n:4d} sesiones ({pct:.1f}%)")

n_any = int(df_all["any_flag"].sum())
n_imp = int(df_all["flag_impossible"].sum())
print(f"\n  {'TOTAL con algun flag':44s}: {n_any:4d} sesiones ({n_any/n_with_hr*100:.1f}%)")
print(f"  {'  -> se eliminaran (imposibles)':44s}: {n_imp:4d} sesiones")

# ── Diagnóstico por atleta ───────────────────────────────────────────────────
hr_rows = df_all[df_all["avg_hr"].notna()]
aq = hr_rows.groupby("cedula").agg(
    n_hr       =("avg_hr",         "count"),
    hr_mean    =("avg_hr",         "mean"),
    hr_min     =("avg_hr",         "min"),
    hr_max     =("avg_hr",         "max"),
    pct_mean   =("pct_hr_max",     "mean"),
    fl_low_hr  =("flag_low_hr",    "sum"),
    fl_hi_hr   =("flag_high_hr",   "sum"),
    fl_lo_pct  =("flag_low_pct",   "sum"),
    fl_hi_pct  =("flag_high_pct",  "sum"),
    fl_flat    =("flag_flat",      "sum"),
    fl_imp     =("flag_impossible", "sum"),
    n_any      =("any_flag",       "sum"),
).reset_index()

aq["pct_flagged"] = (aq["n_any"] / aq["n_hr"] * 100).round(1)
aq["athlete_id"]  = aq["cedula"].map(cedula_to_idx)


def quality_tier(row):
    if row["fl_imp"] > 0:          return "ROJO   - Datos imposibles"
    if row["pct_flagged"] > 30:    return "NARANJA- Alta tasa flags"
    if row["pct_flagged"] > 10:    return "AMARILLO-Flags moderados"
    return "VERDE  - Calidad OK"


aq["quality"] = aq.apply(quality_tier, axis=1)
aq_sorted = aq.sort_values("pct_flagged", ascending=False)

print("\n" + "=" * 62)
print("DIAGNOSTICO DE CALIDAD HR POR ATLETA")
print("=" * 62)
disp = aq_sorted[["athlete_id", "n_hr", "hr_mean", "hr_min", "hr_max",
                   "pct_mean", "pct_flagged", "fl_imp", "quality"]].copy()
disp.columns = ["ID", "N_HR", "HR_med", "HR_min", "HR_max",
                "%FCmax", "%flag", "N_impos", "Calidad"]
print(disp.round(1).to_string(index=False))


In [ ]:
# ── Visualizaciones de calidad HR ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Auditoria calidad HR — Diagnostico por atleta", fontsize=12, fontweight="bold")

TIER_COLORS = {
    "VERDE  - Calidad OK":       "#22C55E",
    "AMARILLO-Flags moderados":  "#EAB308",
    "NARANJA- Alta tasa flags":  "#F97316",
    "ROJO   - Datos imposibles": "#EF4444",
}
TIER_ICONS = {
    "VERDE  - Calidad OK":       "OK",
    "AMARILLO-Flags moderados":  "~",
    "NARANJA- Alta tasa flags":  "!",
    "ROJO   - Datos imposibles": "X",
}

# Panel 1 — % flagged por atleta (barras horizontales, coloreadas por tier)
ax = axes[0]
bar_colors = [TIER_COLORS.get(q, SLATE) for q in aq_sorted["quality"]]
labels_y   = [f"A{int(a)}" for a in aq_sorted["athlete_id"]]
ax.barh(labels_y, aq_sorted["pct_flagged"], color=bar_colors, alpha=0.85, edgecolor="white")
ax.axvline(10, color="orange", ls="--", lw=1.5, alpha=0.7, label="10% umbral moderado")
ax.axvline(30, color="red",    ls="--", lw=1.5, alpha=0.7, label="30% umbral alto")
ax.set_xlabel("% sesiones con algun flag HR")
ax.set_title("% Flags por atleta\n(coloreado por quality tier)")
ax.legend(fontsize=8)
for i, (_, row) in enumerate(aq_sorted.iterrows()):
    icon = TIER_ICONS.get(row["quality"], "?")
    ax.text(row["pct_flagged"] + 0.5, i, icon, va="center", fontsize=10,
            color=TIER_COLORS.get(row["quality"], SLATE))

# Panel 2 — Heatmap de tipos de flag por atleta (N sesiones)
ax = axes[1]
hm_cols = ["fl_low_hr", "fl_hi_hr", "fl_lo_pct", "fl_hi_pct", "fl_flat", "fl_imp"]
hm_lbls = ["HR baja", "HR alta", "%FC bajo", "%FC alto", "Plano", "Imposible"]
hm_data = aq_sorted.set_index("athlete_id")[hm_cols].copy()
hm_data.index = [f"A{int(i)}" for i in hm_data.index]
hm_data.columns = hm_lbls
vmax = hm_data.values.max() if hm_data.values.max() > 0 else 1
sns.heatmap(hm_data, ax=ax, cmap="YlOrRd", annot=True, fmt=".0f",
            linewidths=0.5, linecolor="white", vmin=0, vmax=vmax,
            cbar_kws={"label": "N sesiones flagged"})
ax.set_title("Tipos de flag por atleta\n(N sesiones)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=35)

# Panel 3 — Distribución HR con zonas de calidad marcadas
ax = axes[2]
hr_vals = df_all["avg_hr"].dropna()
if len(hr_vals) > 0:
    xmin_h = max(0, hr_vals.min() - 5)
    xmax_h = hr_vals.max() + 5
    ax.hist(hr_vals, bins=40, color=NAVY, alpha=0.7, edgecolor="white",
            label=f"Todas ({len(hr_vals)} sesiones)")
    ax.axvspan(xmin_h,     HR_MIN_RUN, alpha=0.18, color="red",
               label=f"Sospechosa (<{HR_MIN_RUN})")
    ax.axvspan(HR_MAX_RUN, xmax_h,     alpha=0.18, color="orange",
               label=f"Sospechosa (>{HR_MAX_RUN})")
    ax.axvline(HR_MIN_RUN, color="red",    lw=1.5, ls="--")
    ax.axvline(HR_MAX_RUN, color="orange", lw=1.5, ls="--")
    ax.axvline(hr_vals.median(), color=CRIMSON, lw=2,
               label=f"Mediana {hr_vals.median():.0f} bpm")
    ax.set_xlim(xmin_h, xmax_h)
else:
    ax.text(0.5, 0.5, "Sin datos HR", ha="center", va="center", transform=ax.transAxes)
ax.set_xlabel("FC media de sesion (bpm)")
ax.set_ylabel("N sesiones")
ax.set_title("Distribucion HR global\ncon umbrales de calidad")
ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OUT_DIR / "nb13_fig2b_hr_quality_audit.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {OUT_DIR / 'nb13_fig2b_hr_quality_audit.png'}")


In [ ]:
# ── Aplicar limpieza → df_clean (usar en A.4+ y en todos los modelos) ────────

# Paso 1: eliminar únicamente los fisiológicamente imposibles
df_clean = df_all[~df_all["flag_impossible"]].copy()

# Paso 2: windsorizar avg_hr al intervalo [p2, p98] del dataset limpio
p02 = df_clean["avg_hr"].quantile(0.02)
p98 = df_clean["avg_hr"].quantile(0.98)
df_clean["avg_hr_w"] = df_clean["avg_hr"].clip(p02, p98)

# Paso 3: recalcular %FCmax y zona HR con la HR winsorizada
df_clean["pct_hr_max_w"] = np.where(
    df_clean["hr_max_est"].notna(),
    df_clean["avg_hr_w"] / df_clean["hr_max_est"],
    np.nan,
)
df_clean["zona_hr_w"] = df_clean["pct_hr_max_w"].apply(zona_hr)

# ── Resumen antes vs después ─────────────────────────────────────────────────
n_orig    = len(df_all)
n_clean   = len(df_clean)
n_removed = n_orig - n_clean
n_hr_orig  = int(df_all["avg_hr"].notna().sum())
n_hr_clean = int(df_clean["avg_hr_w"].notna().sum())
n_susp     = int(df_clean["any_flag"].sum())
n_ok       = int((~df_clean["any_flag"]).sum())

print("RESULTADO DE LIMPIEZA HR")
print("=" * 52)
print(f"  Sesiones totales (antes):          {n_orig:6d}")
print(f"  Eliminadas (HR imposible):         {n_removed:6d}")
print(f"  Sesiones en df_clean:              {n_clean:6d}")
print()
print(f"  Con HR (antes):                    {n_hr_orig:6d}")
print(f"  Con HR (despues, winsorizada):     {n_hr_clean:6d}")
print()
print(f"  Rango windsorizado: [{p02:.0f}, {p98:.0f}] bpm  (p2-p98)")
print(f"  HR media original:     {df_all['avg_hr'].mean():.1f} bpm")
print(f"  HR media winsorizada:  {df_clean['avg_hr_w'].mean():.1f} bpm")
print()
print(f"  Sesiones con flags (windsor., NO eliminadas): {n_susp} ({n_susp/n_clean*100:.1f}%)")
print(f"  Sesiones completamente limpias (sin flag):    {n_ok}  ({n_ok/n_clean*100:.1f}%)")
print()
print("Variables en df_clean para modelar:")
print("  avg_hr_w     → FC media winsorizada  [USAR EN MODELOS]")
print("  pct_hr_max_w → %FCmax recalculada    [USAR EN MODELOS]")
print("  zona_hr_w    → Zona HR (1-5)         [USAR EN MODELOS]")
print("  avg_hr       → FC media original     (conservada para diagnóstico)")
print()
print(">>> IMPORTANTE: usar df_clean y avg_hr_w en las secciones A.4+ y en todos los modelos.")

# Actualizar df_hr_clean para las secciones siguientes
df_hr_clean = df_clean[
    df_clean["avg_hr_w"].notna() & df_clean["pace_min_km"].notna()
].copy()
print(f"\ndf_hr_clean: {len(df_hr_clean)} sesiones con HR winsorizada + pace validos")
print(f"             {df_hr_clean['cedula'].nunique()} atletas representados")


## A · 4 — Relación FC–Ritmo: señal central del Nivel 2

In [ ]:
from scipy.stats import pearsonr, spearmanr

df_hr_clean = df_hr.dropna(subset=["avg_hr", "pace_min_km", "pct_hr_max"])

r_pearson, p_pearson = pearsonr(df_hr_clean["avg_hr"], df_hr_clean["pace_min_km"])
r_spearman, p_spearman = spearmanr(df_hr_clean["avg_hr"], df_hr_clean["pace_min_km"])
r_pct, p_pct = pearsonr(df_hr_clean["pct_hr_max"], df_hr_clean["pace_min_km"])

print(f"Correlación FC media ↔ ritmo:")
print(f"  Pearson r = {r_pearson:.3f}  (p={p_pearson:.2e})")
print(f"  Spearman ρ = {r_spearman:.3f}  (p={p_spearman:.2e})")
print(f"\nCorrelación %FCmax ↔ ritmo (ritmo relativo):")
print(f"  Pearson r = {r_pct:.3f}  (p={p_pct:.2e})")
print(f"\nInterpretación:")
print(f"  {'Relación positiva significativa' if r_pearson > 0.1 and p_pearson < 0.05 else 'Relación débil'} entre FC media y ritmo")
print(f"  (A ritmo más lento = sesiones de mayor FC? Mixto: afectan distancia, fatiga, terreno)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Relación FC–Ritmo en atletas RUNA (actividad por actividad)', fontsize=12, fontweight='bold')

# 1. Scatter global FC vs ritmo (coloreado por atleta)
ax = axes[0]
n_athletes = df_hr_clean["athlete_id"].nunique()
colors_scatter = plt.cm.tab20(np.linspace(0, 1, n_athletes))
for i, (aid, grp) in enumerate(df_hr_clean.groupby("athlete_id")):
    ax.scatter(grp["avg_hr"], grp["pace_min_km"], alpha=0.4, s=15,
               color=colors_scatter[i % len(colors_scatter)], label=f'A{aid}' if n_athletes <= 15 else None)
# Línea de tendencia global
from numpy.polynomial.polynomial import polyfit as polyfit_np
coefs = np.polyfit(df_hr_clean["avg_hr"], df_hr_clean["pace_min_km"], 1)
x_line = np.linspace(df_hr_clean["avg_hr"].min(), df_hr_clean["avg_hr"].max(), 100)
ax.plot(x_line, np.polyval(coefs, x_line), color='black', lw=2, ls='--',
        label=f'Tendencia (r={r_pearson:.2f})')
ax.set_xlabel('FC media (bpm)')
ax.set_ylabel('Ritmo (min/km)')
ax.set_title(f'FC media vs Ritmo\n(N={len(df_hr_clean)} actividades)')
ax.legend(fontsize=7, ncol=2 if n_athletes > 10 else 1)

# 2. Scatter %FCmax vs ritmo
ax = axes[1]
ax.scatter(df_hr_clean["pct_hr_max"] * 100, df_hr_clean["pace_min_km"],
           alpha=0.4, s=15, color=NAVY)
coefs2 = np.polyfit(df_hr_clean["pct_hr_max"], df_hr_clean["pace_min_km"], 1)
x2 = np.linspace(df_hr_clean["pct_hr_max"].min(), df_hr_clean["pct_hr_max"].max(), 100)
ax.plot(x2 * 100, np.polyval(coefs2, x2), color=CRIMSON, lw=2, ls='--',
        label=f'Tendencia (r={r_pct:.2f})')
ax.set_xlabel('% FCmax estimada')
ax.set_ylabel('Ritmo (min/km)')
ax.set_title(f'%FCmax vs Ritmo\n(N={len(df_hr_clean)})')
ax.legend(fontsize=9)

# 3. FC media por zona vs ritmo medio de la zona
ax = axes[2]
zona_summary = df_hr_clean.groupby("zona_hr").agg(
    n=("avg_hr","count"), avg_hr=("avg_hr","mean"), avg_pace=("pace_min_km","mean")
).reset_index()
ax.scatter(zona_summary["avg_hr"], zona_summary["avg_pace"],
           s=zona_summary["n"]*2, color=CRIMSON, alpha=0.8, zorder=5)
for _, row in zona_summary.iterrows():
    ax.annotate(f'Z{int(row["zona_hr"])}\n(n={int(row["n"])})',
                (row["avg_hr"], row["avg_pace"]),
                textcoords='offset points', xytext=(5, 5), fontsize=9)
ax.set_xlabel('FC media de la zona (bpm)')
ax.set_ylabel('Ritmo medio (min/km)')
ax.set_title('Resumen por zona de intensidad\n(tamaño = N actividades)')

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig3_hr_vs_ritmo.png', dpi=150, bbox_inches='tight')
plt.show()

## A · 5 — Variabilidad inter-atleta: por qué necesitamos el Nivel 2

In [ ]:
# Correlación HR-ritmo POR ATLETA — muestra heterogeneidad inter-atleta
corrs_by_athlete = []
for ced, grp in df_hr_clean.groupby("cedula"):
    if len(grp) >= 5:
        r, p = pearsonr(grp["avg_hr"], grp["pace_min_km"])
        corrs_by_athlete.append({
            "athlete_id": cedula_to_idx[ced],
            "n": len(grp),
            "r_hr_pace": r,
            "p": p,
            "avg_pace": grp["pace_min_km"].mean(),
            "avg_hr": grp["avg_hr"].mean(),
        })

df_corrs = pd.DataFrame(corrs_by_athlete).sort_values("r_hr_pace")

print("Correlación FC↔Ritmo por atleta (>= 5 actividades con HR):")
print(df_corrs[["athlete_id","n","r_hr_pace","p","avg_pace","avg_hr"]].to_string(index=False))
print(f"\nRango: r = [{df_corrs['r_hr_pace'].min():.2f}, {df_corrs['r_hr_pace'].max():.2f}]")
print(f"Mediana: r = {df_corrs['r_hr_pace'].median():.2f}")
print(f"\n→ Alta variabilidad inter-atleta justifica personalización (Nivel 2)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Variabilidad inter-atleta — por qué personalizar', fontsize=12, fontweight='bold')

# Izquierda: barras de correlación por atleta
ax = axes[0]
colors_bar = [CRIMSON if r < 0 else NAVY for r in df_corrs["r_hr_pace"]]
bars = ax.barh([f'A{aid}' for aid in df_corrs["athlete_id"]],
               df_corrs["r_hr_pace"], color=colors_bar, alpha=0.8)
ax.axvline(0, color='black', lw=1)
ax.axvline(df_corrs["r_hr_pace"].median(), color='orange', lw=2, ls='--',
           label=f'Mediana r={df_corrs["r_hr_pace"].median():.2f}')
ax.set_xlabel('Pearson r (FC ↔ Ritmo)')
ax.set_title('Correlación FC-Ritmo por atleta\n(mismo feature, respuesta diferente)')
ax.legend(fontsize=9)

# Derecha: curvas FC-ritmo por atleta (regresión lineal por atleta)
ax = axes[1]
x_global = np.linspace(df_hr_clean["avg_hr"].min(), df_hr_clean["avg_hr"].max(), 100)
for i, (ced, grp) in enumerate(df_hr_clean.groupby("cedula")):
    if len(grp) >= 5:
        coef = np.polyfit(grp["avg_hr"], grp["pace_min_km"], 1)
        y_pred = np.polyval(coef, x_global)
        mask = (x_global >= grp["avg_hr"].min()) & (x_global <= grp["avg_hr"].max())
        ax.plot(x_global[mask], y_pred[mask], alpha=0.7, lw=1.5,
                color=colors_scatter[i % len(colors_scatter)],
                label=f'A{cedula_to_idx[ced]}')

ax.set_xlabel('FC media (bpm)')
ax.set_ylabel('Ritmo predicho (min/km)')
ax.set_title('Pendientes FC→Ritmo por atleta\n(cada línea = un atleta)')
if df_hr_clean["athlete_id"].nunique() <= 15:
    ax.legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig4_variabilidad_interatleta.png', dpi=150, bbox_inches='tight')
plt.show()

## A · 6 — Heatmap semanal: evolución de la carga por atleta

In [ ]:
if df_wf.empty:
    print("Sin weekly_features en Supabase — saltando heatmap")
else:
    df_wf["athlete_id"] = df_wf["cedula"].map(cedula_to_idx)
    df_wf["week_label"] = df_wf["week_start"].dt.strftime("%Y-%m-%d")

    # Pivot: atletas en eje Y, semanas en eje X, valor = CTL o total_km
    pivot_ctl = df_wf.pivot_table(index="athlete_id", columns="week_label", values="ctl", aggfunc="mean")
    pivot_km = df_wf.pivot_table(index="athlete_id", columns="week_label", values="total_km", aggfunc="sum")

    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    fig.suptitle('Evolución semanal por atleta (RUNA)', fontsize=13, fontweight='bold')

    # CTL heatmap
    ax = axes[0]
    # Mostrar solo las últimas 24 semanas para que sea legible
    cols = pivot_ctl.columns[-24:]
    sns.heatmap(pivot_ctl[cols], ax=ax, cmap='YlOrRd', cbar_kws={'label': 'CTL'},
                linewidths=0.5, linecolor='white', annot=False,
                yticklabels=[f'A{i}' for i in pivot_ctl.index])
    ax.set_title('CTL (Carga Crónica) por atleta y semana')
    ax.set_xlabel('Semana')
    ax.set_ylabel('Atleta')
    # Rotar labels del eje X
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)

    # km heatmap
    ax = axes[1]
    sns.heatmap(pivot_km[cols], ax=ax, cmap='Blues', cbar_kws={'label': 'km/semana'},
                linewidths=0.5, linecolor='white', annot=False,
                yticklabels=[f'A{i}' for i in pivot_km.index])
    ax.set_title('Volumen semanal (km) por atleta')
    ax.set_xlabel('Semana')
    ax.set_ylabel('Atleta')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'nb13_fig5_heatmap_semanal.png', dpi=150, bbox_inches='tight')
    plt.show()

## A · 7 — Primer modelo Ridge: baseline Nivel 2 con LOAO-CV

**LOAO-CV** (Leave-One-Athlete-Out Cross-Validation): para cada atleta en el conjunto
de test, entrenamos con todos los demás. Esto simula exactamente la situación de
predecir el ritmo de un atleta nuevo dado su FC + perfil.

Esto es el **baseline del Nivel 2**: si Ridge ya supera la mediana global, hay señal.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, r2_score

# ── Dataset para modelado ────────────────────────────────────────────────────
# Solo atletas con HR, perfil de edad/sexo
df_model = df_hr_clean.dropna(subset=["avg_hr","pct_hr_max","zona_hr","pace_min_km","age","sex"]).copy()
df_model["sex_bin"] = (df_model["sex"].str.upper() == "M").astype(float)
df_model["zona_hr"] = df_model["zona_hr"].astype(float)

FEATURES_V1 = ["avg_hr", "pct_hr_max", "zona_hr", "distance_km",
               "duration_min", "elevation_m", "age", "sex_bin"]
# Rellenar elevation NaN con 0
for col in FEATURES_V1:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce").fillna(0)

TARGET = "pace_min_km"
GROUP = "cedula"

# Filtrar atletas con al menos 5 actividades en el modelo
counts = df_model.groupby("cedula").size()
eligible_cedulas = counts[counts >= 5].index
df_model = df_model[df_model["cedula"].isin(eligible_cedulas)].copy()

X = df_model[FEATURES_V1].values
y = df_model[TARGET].values
groups = df_model[GROUP].values

print(f"Dataset LOAO-CV:")
print(f"  Actividades: {len(df_model)}")
print(f"  Atletas: {len(np.unique(groups))}")
print(f"  Features: {FEATURES_V1}")
print(f"  Target: {TARGET} (ritmo min/km)")
print(f"  Rango target: [{y.min():.2f}, {y.max():.2f}]")

In [ ]:
logo = LeaveOneGroupOut()
preds_ridge = np.zeros_like(y, dtype=float)
preds_naive = np.zeros_like(y, dtype=float)
mae_by_athlete = {}

for tr_idx, te_idx in logo.split(X, y, groups):
    scaler = StandardScaler().fit(X[tr_idx])
    model = Ridge(alpha=1.0)
    model.fit(scaler.transform(X[tr_idx]), y[tr_idx])
    preds_ridge[te_idx] = model.predict(scaler.transform(X[te_idx]))
    preds_naive[te_idx] = np.median(y[tr_idx])  # baseline: mediana del training

    # MAE por atleta (para el plot por atleta)
    ced = np.unique(groups[te_idx])[0]
    mae_by_athlete[cedula_to_idx[ced]] = {
        "mae_ridge": mean_absolute_error(y[te_idx], preds_ridge[te_idx]) * 60,  # a sec/km
        "mae_naive": mean_absolute_error(y[te_idx], preds_naive[te_idx]) * 60,
        "n": len(te_idx),
    }

# Métricas globales
mae_ridge_s = mean_absolute_error(y, preds_ridge) * 60
mae_naive_s = mean_absolute_error(y, preds_naive) * 60
r2_ridge = r2_score(y, preds_ridge)
r2_naive = r2_score(y, preds_naive)

print("=" * 50)
print("RESULTADOS LOAO-CV — Nivel 2 baseline")
print("=" * 50)
print(f"  Baseline (mediana):   MAE = {mae_naive_s:.1f} sec/km   R² = {r2_naive:.3f}")
print(f"  Ridge (V1):           MAE = {mae_ridge_s:.1f} sec/km   R² = {r2_ridge:.3f}")
print(f"  Mejora vs baseline:   {mae_naive_s - mae_ridge_s:+.1f} sec/km ({(1 - mae_ridge_s/mae_naive_s)*100:.1f}%)")
print()
# MAE por atleta
df_mae = pd.DataFrame(mae_by_athlete).T.reset_index().rename(columns={"index": "athlete_id"})
df_mae = df_mae.sort_values("mae_ridge")
print("MAE por atleta (sec/km):")
print(df_mae.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('LOAO-CV — Ridge Nivel 2 vs Baseline', fontsize=12, fontweight='bold')

# 1. Predicho vs Real
ax = axes[0]
ax.scatter(y, preds_ridge, alpha=0.5, s=20, color=NAVY, label='Ridge V1')
ax.scatter(y, preds_naive, alpha=0.3, s=10, color=SLATE, label='Baseline (mediana)')
min_v, max_v = min(y.min(), preds_ridge.min()), max(y.max(), preds_ridge.max())
ax.plot([min_v, max_v], [min_v, max_v], 'k--', lw=1.5, label='Perfecto')
ax.set_xlabel('Ritmo real (min/km)')
ax.set_ylabel('Ritmo predicho (min/km)')
ax.set_title(f'Predicho vs Real\nRidge MAE={mae_ridge_s:.0f} s/km · Baseline={mae_naive_s:.0f} s/km')
ax.legend(fontsize=9)

# 2. MAE por atleta: Ridge vs baseline
ax = axes[1]
x_pos = np.arange(len(df_mae))
w = 0.35
ax.bar(x_pos - w/2, df_mae["mae_naive"], w, color=SLATE, alpha=0.7, label='Baseline')
ax.bar(x_pos + w/2, df_mae["mae_ridge"], w, color=CRIMSON, alpha=0.8, label='Ridge V1')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'A{int(a)}' for a in df_mae["athlete_id"]], rotation=45, ha='right')
ax.set_ylabel('MAE (sec/km)')
ax.set_title('MAE por atleta (LOAO-CV)')
ax.legend(fontsize=9)
ax.axhline(mae_ridge_s, color=CRIMSON, ls='--', lw=1.5, alpha=0.7, label=f'Media Ridge={mae_ridge_s:.0f}')

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig6_loao_cv_ridge.png', dpi=150, bbox_inches='tight')
plt.show()

## A · 8 — Importancia de features (coeficientes Ridge)

In [ ]:
# Entrenar Ridge en dataset completo para ver coeficientes (interpretabilidad)
scaler_full = StandardScaler().fit(X)
ridge_full = Ridge(alpha=1.0).fit(scaler_full.transform(X), y)

coef_df = pd.DataFrame({
    "feature": FEATURES_V1,
    "coef": ridge_full.coef_,
    "abs_coef": np.abs(ridge_full.coef_),
}).sort_values("abs_coef", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors_coef = [CRIMSON if c < 0 else NAVY for c in coef_df["coef"]]
ax.barh(coef_df["feature"], coef_df["coef"], color=colors_coef, alpha=0.85)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Coeficiente Ridge (espacio estandarizado)")
ax.set_title("Importancia de features — Ridge Nivel 2\n(entrenado en dataset completo)")

# Anotaciones
for _, row in coef_df.iterrows():
    ax.text(row["coef"] + (0.002 if row["coef"] >= 0 else -0.002),
            row.name, f'{row["coef"]:.3f}', va='center',
            ha='left' if row["coef"] >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig7_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nIntercept: {ridge_full.intercept_:.3f}")
print(f"\nInterpretación:")
for _, row in coef_df.sort_values("abs_coef", ascending=False).iterrows():
    direction = "↑ pace (más lento)" if row["coef"] > 0 else "↓ pace (más rápido)"
    print(f"  {row['feature']:20s}: coef={row['coef']:+.3f} → {direction}")

---

# PARTE B — Arquitectura del Nivel 2 (cohort-level generalizable)

## B · 1 — Definición formal y separación N2 / N3

### Corrección arquitectónica (2026-05-12)

La versión anterior de NB13 incluía CTL/ATL/ACWR como features del Nivel 2. Esto es incorrecto:
- **CTL/ATL/ACWR** son EWMA sobre el historial acumulado del propio atleta → requieren historial longitudinal → no disponibles para un atleta nuevo → contradicen la etiqueta "generalizable".
- **Fix:** CTL/ATL/ACWR migran al N3 (longitudinal individual). El N2 solo usa variables disponibles en la primera sesión de cualquier atleta.

### Nivel 2 — Cohort-level generalizable
```
Pregunta: dado el FC de una sesión y el perfil demográfico del atleta,
          ¿cuál es el ritmo esperado?
Aplica a: cualquier atleta (nuevo o existente)
Inputs:   una sesión (avg_hr, distance_km, elevation) + perfil (age, sex, vdot)
          + pred_nivel1 (stacking feature del prior poblacional)
Target:   pace_min_km
```

**Hipótesis formal:**
- **H₀:** MAE_N2 ≥ MAE_N1_LOAO (sin ganancia sobre el prior poblacional)
- **H₁:** MAE_N2 < MAE_N1_LOAO (la cohorte RUNA aporta señal adicional)

### Nivel 3 — Longitudinal individual (diseño para NB14)
```
Pregunta: dado el historial acumulado de *este* atleta y su carga reciente,
          ¿cuál es su ritmo esperado *ahora*?
Aplica a: atletas con ≥3 carreras propias registradas
Inputs:   todo N2 + CTL + ATL + TSB + ACWR + own_race_best_pace + n_races
Target:   pace_min_km  (mismo target, más contexto individual)
```

### Relación N1 → N2 → N3 (stacking jerárquico)
```
N1: prior_poblacional(hr, pct_hrmax, zona, gender, log_duration)
    → pred_nivel1  (feature de entrada para N2)

N2: cohort_model(pred_nivel1, age, sex, vdot, avg_hr, log_distance, ...)
    → pred_nivel2  (feature de entrada para N3)

N3: individual_model(pred_nivel2, CTL, ATL, ACWR, own_race_pace, ...)
    → pred_nivel3  (predicción final personalizada)
```

> Referencia metodológica: Wolpert (1992) — Stacked Generalization.

## B · 2 — Feature set N2: construcción y cobertura

### Variables del Nivel 2 (cohort-level, sin CTL/ATL)

| Categoría | Variable | Fuente | Justificación | Cobertura esperada |
|-----------|----------|--------|---------------|-------------------|
| **Stacking N1** | `pred_nivel1` | modelo N1 | Prior poblacional (Wolpert 1992) | ~100% |
| **Demografía** | `age` | athlete_profiles | Control de nivel fisiológico | ~85% |
| | `sex_bin` | athlete_profiles | Diferencias fisiológicas M/F | ~85% |
| | `vdot_estimated` | PR declarado → Daniels | Nivel de rendimiento del atleta | ~60% |
| **Intensidad sesión** | `avg_hr_w` | activities.raw (windsorizado) | Señal central del esfuerzo | ~70% |
| | `pct_hr_max_w` | calculado (Tanaka) | Intensidad relativa a su FCmax | ~65% |
| | `zona_hr_w` | calculado | Categoría de esfuerzo (1-5) | ~65% |
| | `fcmax_obs` | max(max_hr) por atleta | FCmax real observada | ~70% |
| **Morfología sesión** | `log_distance_km` | activities | Distancia (NO log_duration: evita circularidad) | ~100% |
| | `elevation_gain_m` | activities | Terreno (afecta ritmo) | ~60% |
| | `has_elevation` | activities | Flag binario desnivel | ~100% |
| | `avg_cadence` | activities.raw | Eficiencia de zancada | ~55% |
| **Temporalidad** | `day_of_week_sin/cos` | activity_date | Estacionalidad semanal | ~100% |

**Variables excluidas y por qué:**
- ~~`log_duration`~~: circular — pace = distance/duration → duración no disponible al predecir para sesión futura
- ~~`CTL/ATL/TSB/ACWR`~~: requieren historial acumulado → van a N3
- ~~`PRs declarados`~~: sesgo de recuerdo, baja confiabilidad → reemplazados por `vdot_estimated`

## B · 3 — Construcción del dataset N2 y stacking del Nivel 1

In [ ]:
import pickle
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, r2_score

# ── 1. Cargar modelo Nivel 1 para stacking ────────────────────────────────
N1_PKL = BASE / 'ml/notebooks/outputs/nb12/nivel1_prior_poblacional_FULL_v5.pkl'
n1_model_n1 = None
n1_scaler_n1 = None
n1_features_n1 = None

if N1_PKL.exists():
    with open(N1_PKL, 'rb') as f:
        n1_art = pickle.load(f)
    n1_model_n1   = n1_art.get('model')
    n1_scaler_n1  = n1_art.get('scaler')
    n1_features_n1 = n1_art.get('features')
    print(f"N1 model cargado: {type(n1_model_n1).__name__}")
    print(f"N1 features:      {n1_features_n1}")
else:
    print(f"WARN: N1 pkl no encontrado en {N1_PKL}")
    print("pred_nivel1 será NaN → se imputará con mediana")

# ── 2. Construir features N2 sobre df_clean ──────────────────────────────
df_n2 = df_clean.copy()

# 2a. log_distance_km (no log_duration — circular)
df_n2["log_distance_km"] = np.log1p(df_n2["distance_km"].clip(lower=0.01))

# 2b. has_elevation + elevation_gain_m
df_n2["has_elevation"] = (df_n2["elevation_m"].notna() & (df_n2["elevation_m"] > 0)).astype(float)
df_n2["elevation_gain_m"] = df_n2["elevation_m"].fillna(0.0)

# 2c. avg_cadence (de raw si no existe ya)
if "avg_cadence" not in df_n2.columns:
    df_n2["avg_cadence"] = df_n2.get("average_cadence", np.nan)

# 2d. fcmax_obs por atleta: max de max_hr en sus actividades
fcmax_by_athlete = (
    df_clean.groupby("cedula")["max_hr"]
    .max()
    .rename("fcmax_obs")
    .reset_index()
)
df_n2 = df_n2.merge(fcmax_by_athlete, on="cedula", how="left")

# 2e. vdot_estimated desde PRs declarados (Jack Daniels)
# Fórmula simplificada: VDOT = velocidad_m_min × (0.000104×v² + 0.182258×v - 4.602985)
# donde v = distancia_km / tiempo_min
def vdot_from_time(dist_km, time_min):
    """VDOT estimado de Jack Daniels a partir de distancia y tiempo."""
    if pd.isna(dist_km) or pd.isna(time_min) or time_min <= 0:
        return np.nan
    v = dist_km * 1000 / time_min  # m/min
    pct_vo2 = 0.8 + 0.1894393 * np.exp(-0.012778 * time_min) + 0.2989558 * np.exp(-0.1932605 * time_min)
    vo2 = -4.60 + 0.182258 * v + 0.000104 * v**2
    return vo2 / pct_vo2

# Intentar obtener PRs del formulario → athletes_profiles
pr_raw = (client.table("athlete_profiles")
          .select("cedula,pr_5k,pr_10k,pr_21k,pr_42k")
          .execute()).data or []
df_prs = pd.DataFrame(pr_raw)

if not df_prs.empty:
    def parse_time_min(t):
        """Convierte 'HH:MM:SS' o 'MM:SS' a minutos."""
        if pd.isna(t): return np.nan
        try:
            parts = str(t).split(":")
            if len(parts) == 3:
                return int(parts[0])*60 + int(parts[1]) + int(parts[2])/60
            elif len(parts) == 2:
                return int(parts[0]) + int(parts[1])/60
        except:
            pass
        return np.nan

    vdot_rows = []
    for _, row in df_prs.iterrows():
        candidates = []
        for dist_km, col in [(5, 'pr_5k'), (10, 'pr_10k'), (21.1, 'pr_21k'), (42.2, 'pr_42k')]:
            v = vdot_from_time(dist_km, parse_time_min(row.get(col)))
            if not np.isnan(v):
                candidates.append(v)
        # VDOT del mejor PR (más alto = más informativo del nivel real)
        vdot_rows.append({'cedula': row['cedula'],
                          'vdot_estimated': max(candidates) if candidates else np.nan})
    df_vdot = pd.DataFrame(vdot_rows)
    df_n2 = df_n2.merge(df_vdot, on="cedula", how="left")
    coverage_vdot = df_n2["vdot_estimated"].notna().mean() * 100
    print(f"vdot_estimated: {df_n2['vdot_estimated'].notna().sum()} sesiones, cobertura {coverage_vdot:.1f}%")
else:
    df_n2["vdot_estimated"] = np.nan
    print("WARN: sin datos de PRs en athlete_profiles")

# 2f. Temporalidad
df_n2["dow"] = df_n2["activity_date"].dt.dayofweek
df_n2["month"] = df_n2["activity_date"].dt.month
df_n2["dow_sin"] = np.sin(2 * np.pi * df_n2["dow"] / 7)
df_n2["dow_cos"] = np.cos(2 * np.pi * df_n2["dow"] / 7)
df_n2["month_sin"] = np.sin(2 * np.pi * df_n2["month"] / 12)
df_n2["month_cos"] = np.cos(2 * np.pi * df_n2["month"] / 12)

# 2g. sex_bin
df_n2["sex_bin"] = (df_n2["sex"].str.upper() == "M").astype(float)

# ── 3. Generar pred_nivel1 para stacking ──────────────────────────────────
# N1 features: gender_bin, fcmax_obs, hr_mean, pct_fcmax, zona_num, hr_max_rel, log_duration, dens_hr
if n1_model_n1 is not None:
    df_n2["n1_gender_bin"]  = df_n2["sex_bin"]
    df_n2["n1_fcmax_obs"]   = df_n2["fcmax_obs"].fillna(df_n2["hr_max_est"])
    df_n2["n1_hr_mean"]     = df_n2["avg_hr_w"]
    df_n2["n1_pct_fcmax"]   = df_n2["pct_hr_max_w"]
    df_n2["n1_zona_num"]    = df_n2["zona_hr_w"]
    df_n2["n1_hr_max_rel"]  = (df_n2["max_hr"] / df_n2["n1_fcmax_obs"]).clip(0, 1.1)
    df_n2["n1_log_duration"] = np.log1p(df_n2["duration_sec"].clip(lower=1))
    df_n2["n1_dens_hr"]     = 1.0  # default: Strava watch (continuous HR)

    n1_feat_cols = [
        "n1_gender_bin","n1_fcmax_obs","n1_hr_mean","n1_pct_fcmax",
        "n1_zona_num","n1_hr_max_rel","n1_log_duration","n1_dens_hr"
    ]
    X_n1_input = df_n2[n1_feat_cols].fillna(0).values
    X_n1_scaled = n1_scaler_n1.transform(X_n1_input)
    df_n2["pred_nivel1"] = n1_model_n1.predict(X_n1_scaled)
    print(f"pred_nivel1 generado: range [{df_n2['pred_nivel1'].min():.2f}, {df_n2['pred_nivel1'].max():.2f}] min/km")
else:
    df_n2["pred_nivel1"] = df_n2["pace_min_km"].mean()  # fallback: media global
    print("WARN: pred_nivel1 = media global (N1 pkl no disponible)")

# ── 4. Feature coverage audit ─────────────────────────────────────────────
FEATURES_N2 = [
    "pred_nivel1",
    "age", "sex_bin", "vdot_estimated",
    "avg_hr_w", "pct_hr_max_w", "zona_hr_w", "fcmax_obs",
    "log_distance_km", "elevation_gain_m", "has_elevation", "avg_cadence",
    "dow_sin", "dow_cos",
]

print("\nAUDITORÍA DE COBERTURA — Features N2")
print("=" * 48)
for feat in FEATURES_N2:
    if feat in df_n2.columns:
        cov = df_n2[feat].notna().mean() * 100
        print(f"  {feat:30s}: {cov:5.1f}% ({df_n2[feat].notna().sum()} sesiones)")
    else:
        print(f"  {feat:30s}: ❌ NO EXISTE")

print(f"\nTotal sesiones en df_n2: {len(df_n2)}")
print(f"Atletas únicos:          {df_n2['cedula'].nunique()}")

In [ ]:
# ── Preparar dataset de modelado con imputación y sample weights ─────────

# Filtrar: solo atletas con HR (señal central del modelo)
df_model_n2 = df_n2[
    df_n2["avg_hr_w"].notna() &
    df_n2["pace_min_km"].notna()
].copy()

# Imputación: mediana global para variables con baja cobertura
# (aplicamos ANTES de LOAO-CV para no filtrar demasiadas sesiones)
for feat in FEATURES_N2:
    if feat in df_model_n2.columns:
        median_val = df_model_n2[feat].median()
        df_model_n2[feat] = df_model_n2[feat].fillna(median_val)
    else:
        df_model_n2[feat] = 0.0  # fallback para feature faltante

# Filtrar atletas con ≥5 sesiones después de imputación
counts_n2 = df_model_n2["cedula"].value_counts()
eligible_n2 = counts_n2[counts_n2 >= 5].index
df_model_n2 = df_model_n2[df_model_n2["cedula"].isin(eligible_n2)].copy()

X_n2 = df_model_n2[FEATURES_N2].values.astype(float)
y_n2 = df_model_n2["pace_min_km"].values
groups_n2 = df_model_n2["cedula"].values

# ── Sample weights: w = 1/n_sesiones por atleta ───────────────────────────
n_per_athlete = df_model_n2.groupby("cedula").size()
df_model_n2["sample_weight"] = df_model_n2["cedula"].map(lambda c: 1.0 / n_per_athlete[c])
sample_weights_n2 = df_model_n2["sample_weight"].values

print(f"Dataset N2 para modelado: {len(df_model_n2)} sesiones, {df_model_n2['cedula'].nunique()} atletas")
print(f"Sample weights: min={sample_weights_n2.min():.5f}, max={sample_weights_n2.max():.5f}")
print(f"Features activas: {len(FEATURES_N2)}")

# ── LOAO-CV: Ridge N2 con sample weighting ────────────────────────────────
lao = LeaveOneGroupOut()
preds_n2_ridge = np.zeros_like(y_n2, dtype=float)
preds_n2_naive = np.zeros_like(y_n2, dtype=float)
mae_by_athlete_n2 = {}

# Pre-generar folds (necesario para naïveAutoML en B·4)
loao_folds = list(lao.split(X_n2, y_n2, groups_n2))

for fold_i, (tr_idx, te_idx) in enumerate(loao_folds):
    X_tr, X_te = X_n2[tr_idx], X_n2[te_idx]
    y_tr, y_te = y_n2[tr_idx], y_n2[te_idx]
    w_tr       = sample_weights_n2[tr_idx]

    scaler = StandardScaler().fit(X_tr)
    model  = Ridge(alpha=1.0)
    model.fit(scaler.transform(X_tr), y_tr, sample_weight=w_tr)

    preds_n2_ridge[te_idx] = model.predict(scaler.transform(X_te))
    preds_n2_naive[te_idx] = np.average(y_tr, weights=w_tr)  # baseline: mediana ponderada

    ced = np.unique(groups_n2[te_idx])[0]
    mae_by_athlete_n2[cedula_to_idx.get(ced, ced)] = {
        "mae_ridge_n2": mean_absolute_error(y_te, preds_n2_ridge[te_idx]) * 60,
        "mae_naive": mean_absolute_error(y_te, preds_n2_naive[te_idx]) * 60,
        "n": len(te_idx),
    }

# Comparar también con el N1 predicción pura
preds_n1_on_n2 = df_model_n2["pred_nivel1"].values
mae_n1_pure    = mean_absolute_error(y_n2, preds_n1_on_n2) * 60

mae_n2_ridge   = mean_absolute_error(y_n2, preds_n2_ridge) * 60
mae_n2_naive   = mean_absolute_error(y_n2, preds_n2_naive) * 60
r2_n2_ridge    = r2_score(y_n2, preds_n2_ridge)
r2_n1_pure     = r2_score(y_n2, preds_n1_on_n2)

print("\n" + "=" * 65)
print("RESULTADOS LOAO-CV — Ridge N2 con sample weighting (BASELINE)")
print("=" * 65)
print(f"  Baseline (mediana ponderada):   MAE = {mae_n2_naive:.1f} sec/km")
print(f"  N1 pred directa (stacking ref): MAE = {mae_n1_pure:.1f} sec/km   R² = {r2_n1_pure:.3f}")
print(f"  Ridge N2 (con sample_weight):   MAE = {mae_n2_ridge:.1f} sec/km  R² = {r2_n2_ridge:.3f}")
print()
print(f"  Mejora Ridge N2 vs N1 puro:     {mae_n1_pure - mae_n2_ridge:+.1f} sec/km")
print(f"  Mejora Ridge N2 vs baseline:    {mae_n2_naive - mae_n2_ridge:+.1f} sec/km")
print()
if mae_n2_ridge < mae_n1_pure:
    print(f"  → ✅ H₁ soportada: N2 mejora sobre el prior N1 ({mae_n1_pure - mae_n2_ridge:.1f} sec/km)")
else:
    print(f"  → ❌ H₀ no rechazada: Ridge N2 no mejora sobre N1 puro")
    print(f"     (puede cambiar con naïveAutoML en B·4)")

# MAE por atleta
df_mae_n2 = (pd.DataFrame(mae_by_athlete_n2)
             .T.reset_index()
             .rename(columns={"index": "athlete_id"})
             .sort_values("mae_ridge_n2"))
print("\nMAE por atleta (sec/km, primeros 10):")
print(df_mae_n2.head(10).to_string(index=False))

## B · 4 — naïveAutoML LOAO-CV (7 familias sklearn + AutoML, protocolo anti-leakage)

Mismo protocolo que NB12/N1: **PregenSplitter** + **SplitBasedEvaluator** inyectados al evaluador
interno de naïveautoml. Cada pipeline candidato ve exactamente los mismos folds LOAO-CV que
los 7 modelos sklearn del benchmark → comparación justa y sin fuga.

> **Nota entorno**: naïveautoml requiere Python 3.11 (`conda activate running_coaching`).
> Si no está instalado, la celda hace fallback a comparación manual de los 7 modelos sklearn.

In [ ]:
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

# ── Benchmark: 7 familias sklearn con los mismos folds LOAO-CV ───────────
CANDIDATES = {
    "Ridge":     lambda: Ridge(alpha=1.0),
    "Lasso":     lambda: Lasso(alpha=0.01),
    "ElasticNet": lambda: ElasticNet(alpha=0.01, l1_ratio=0.5),
    "RandomForest": lambda: RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1),
    "GradBoost": lambda: GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=0),
    "HistGB":    lambda: HistGradientBoostingRegressor(max_iter=200, random_state=0),
    "MLP":       lambda: MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=0),
}

results_benchmark = {}

print("=" * 65)
print("BENCHMARK 7 FAMILIAS sklearn — LOAO-CV N2")
print(f"  {len(loao_folds)} folds, {len(X_n2)} sesiones, {len(np.unique(groups_n2))} atletas")
print("=" * 65)

for name, model_fn in CANDIDATES.items():
    preds_fold = np.zeros_like(y_n2, dtype=float)
    maes_per_fold = []
    for tr_idx, te_idx in loao_folds:
        w_tr = sample_weights_n2[tr_idx]
        scaler = StandardScaler().fit(X_n2[tr_idx])
        mdl = model_fn()
        # fit con sample_weight si el modelo lo soporta
        try:
            mdl.fit(scaler.transform(X_n2[tr_idx]), y_n2[tr_idx], sample_weight=w_tr)
        except TypeError:
            mdl.fit(scaler.transform(X_n2[tr_idx]), y_n2[tr_idx])
        preds_fold[te_idx] = mdl.predict(scaler.transform(X_n2[te_idx]))
        maes_per_fold.append(mean_absolute_error(y_n2[te_idx], preds_fold[te_idx]) * 60)

    mae_global = mean_absolute_error(y_n2, preds_fold) * 60
    r2_global  = r2_score(y_n2, preds_fold)
    results_benchmark[name] = {
        "mae": mae_global, "r2": r2_global,
        "mae_per_fold": maes_per_fold,  # usado en Friedman-Nemenyi (B·5)
        "preds": preds_fold.copy(),
    }
    print(f"  {name:15s}: MAE = {mae_global:6.1f} sec/km   R² = {r2_global:+.3f}")

# ── Intentar naïveAutoML si está disponible ───────────────────────────────
automl_result = None
try:
    import naiveautoml
    from naiveautoml.evaluators import SplitBasedEvaluator  # noqa

    class PregenSplitter:
        """Inyecta folds pre-generados al evaluador interno de naïveautoml."""
        def __init__(self, folds):
            self._folds = folds
        def get_n_splits(self, X=None, y=None, groups=None):
            return len(self._folds)
        def split(self, X, y=None, groups=None):
            return iter(self._folds)

    splitter  = PregenSplitter(loao_folds)
    evaluator = SplitBasedEvaluator(splitter)
    automl    = naiveautoml.NaiveAutoML(
        evaluator=evaluator,
        timeout=600,
        show_progress=True,
    )
    automl.fit(X_n2, y_n2)
    preds_automl = np.zeros_like(y_n2, dtype=float)
    maes_automl_fold = []
    for tr_idx, te_idx in loao_folds:
        automl_temp = naiveautoml.NaiveAutoML(evaluator=evaluator, timeout=120, show_progress=False)
        automl_temp.fit(X_n2[tr_idx], y_n2[tr_idx])
        preds_automl[te_idx] = automl_temp.predict(X_n2[te_idx])
        maes_automl_fold.append(mean_absolute_error(y_n2[te_idx], preds_automl[te_idx]) * 60)

    mae_automl = mean_absolute_error(y_n2, preds_automl) * 60
    r2_automl  = r2_score(y_n2, preds_automl)
    results_benchmark["naïveAutoML"] = {
        "mae": mae_automl, "r2": r2_automl,
        "mae_per_fold": maes_automl_fold,
        "preds": preds_automl,
        "best_pipeline": str(automl.best_config),
    }
    automl_result = results_benchmark["naïveAutoML"]
    print(f"\n  {'naïveAutoML':15s}: MAE = {mae_automl:6.1f} sec/km   R² = {r2_automl:+.3f}")
    print(f"  Pipeline ganador: {automl.best_config}")

except ImportError:
    print("\nINFO: naïveautoml no disponible — activar conda env 'running_coaching'")
    print("  Para ejecutar: conda activate running_coaching && jupyter notebook")
    print("  Benchmark sklearn completo disponible arriba ↑")

# Ranking final
print("\n" + "=" * 65)
print("RANKING FINAL (todos los modelos)")
print("=" * 65)
ranking = sorted(results_benchmark.items(), key=lambda x: x[1]["mae"])
for rank, (name, res) in enumerate(ranking, 1):
    bp = res.get("best_pipeline", "")
    print(f"  #{rank:2d} {name:15s}: MAE = {res['mae']:6.1f} sec/km  R² = {res['r2']:+.3f}  {bp}")
print(f"  --- N1 puro (ref):    MAE = {mae_n1_pure:6.1f} sec/km")

## B · 5 — Test Friedman-Nemenyi sobre MAEs por fold LOAO-CV

Mismo protocolo que NB12/N1 (§7 THESIS_CONTEXT.md):
- **Friedman**: ¿hay algún modelo sistemáticamente mejor? (no paramétrico, trabaja con rankings)
- **Nemenyi post-hoc**: ¿cuáles pares difieren significativamente?

In [ ]:
from scipy.stats import friedmanchisquare

# Construir matriz MAE[fold × modelo] — solo modelos con mae_per_fold disponible
fold_maes_dict = {
    name: res["mae_per_fold"]
    for name, res in results_benchmark.items()
    if "mae_per_fold" in res
}

# Asegurar que todos tienen el mismo número de folds
n_folds_common = min(len(v) for v in fold_maes_dict.values())
model_names_frd = list(fold_maes_dict.keys())
mae_matrix = np.array([fold_maes_dict[m][:n_folds_common] for m in model_names_frd]).T
# mae_matrix shape: (n_folds, n_models)

print(f"Friedman-Nemenyi sobre {n_folds_common} folds × {len(model_names_frd)} modelos")
print(f"Modelos: {model_names_frd}\n")

# ── Friedman test ─────────────────────────────────────────────────────────
friedman_stat, friedman_p = friedmanchisquare(*[mae_matrix[:, j] for j in range(mae_matrix.shape[1])])
print(f"Test de Friedman:")
print(f"  χ² = {friedman_stat:.4f}")
print(f"  p  = {friedman_p:.4f}")
if friedman_p < 0.05:
    print(f"  → ✅ p < 0.05: hay diferencias significativas entre modelos")
else:
    print(f"  → ❌ p ≥ 0.05: sin diferencias significativas entre modelos")
    print(f"  (con N={n_folds_common} folds la potencia es limitada — esperable con N<40 atletas)")

# ── Nemenyi post-hoc ──────────────────────────────────────────────────────
try:
    import scikit_posthocs as sp
    nemenyi_p = sp.posthoc_nemenyi_friedman(mae_matrix)
    nemenyi_p.index = model_names_frd
    nemenyi_p.columns = model_names_frd

    print(f"\nNemenyi post-hoc p-valores:")
    print(nemenyi_p.round(3).to_string())

    # Resumen: pares significativos (p < 0.05)
    print("\nPares con diferencia significativa (p < 0.05):")
    found = False
    for i in range(len(model_names_frd)):
        for j in range(i+1, len(model_names_frd)):
            p_pair = nemenyi_p.iloc[i, j]
            if p_pair < 0.05:
                print(f"  {model_names_frd[i]} vs {model_names_frd[j]}: p={p_pair:.4f} ✓")
                found = True
    if not found:
        print("  Ningún par supera el umbral α=0.05")
        print("  (Resultado esperado con N=32 folds — potencia estadística limitada)")

except ImportError:
    print("\nINFO: scikit-posthocs no disponible")
    print("  pip install scikit-posthocs")
    print("  Friedman test ejecutado correctamente — Nemenyi requiere la librería adicional")

# ── Visualización de MAE por fold ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Distribución MAE por fold LOAO-CV — {n_folds_common} folds', fontsize=12, fontweight='bold')

ax = axes[0]
mae_by_model = {name: fold_maes_dict[name][:n_folds_common] for name in model_names_frd}
colors_box = [CRIMSON, NAVY, '#2563EB', '#059669', '#D97706', '#7C3AED', '#DC2626', '#F59E0B']
bp = ax.boxplot(
    [mae_by_model[m] for m in model_names_frd],
    patch_artist=True,
    boxprops=dict(alpha=0.7),
    medianprops=dict(lw=2),
)
for patch, color in zip(bp['boxes'], colors_box[:len(model_names_frd)]):
    patch.set_facecolor(color)
ax.set_xticks(range(1, len(model_names_frd)+1))
ax.set_xticklabels(model_names_frd, rotation=30, ha='right')
ax.set_ylabel('MAE (sec/km) por fold')
ax.set_title('Distribución MAE por fold por modelo')
ax.axhline(mae_n1_pure, color='gray', ls='--', lw=1.5, label=f'N1 puro={mae_n1_pure:.0f}')
ax.legend(fontsize=9)

ax = axes[1]
# Heatmap MAE por fold × modelo (submuestra primeras 15 filas si es muy grande)
n_show = min(n_folds_common, 20)
mat_show = mae_matrix[:n_show]
sns.heatmap(mat_show, ax=ax, cmap='YlOrRd', annot=True, fmt=".0f",
            xticklabels=model_names_frd, yticklabels=[f'fold {i+1}' for i in range(n_show)],
            cbar_kws={"label": "MAE (sec/km)"})
ax.set_title(f'MAE por fold × modelo\n(primeros {n_show} folds)')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig_friedman.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFriedman χ²={friedman_stat:.2f}, p={friedman_p:.4f}, N_folds={n_folds_common}")

## B · 6 — Diagnóstico de errores: por atleta, género y zona HR

In [ ]:
# Seleccionar el modelo ganador (mínimo MAE global)
best_name = min(results_benchmark, key=lambda k: results_benchmark[k]["mae"])
best_preds = results_benchmark[best_name]["preds"]
print(f"Modelo ganador para diagnóstico: {best_name} (MAE={results_benchmark[best_name]['mae']:.1f} sec/km)")

df_diag = df_model_n2.copy()
df_diag["pred_best"] = best_preds
df_diag["error_sec"] = (df_diag["pred_best"] - df_diag["pace_min_km"]) * 60
df_diag["abs_error_sec"] = df_diag["error_sec"].abs()
df_diag["athlete_id"] = df_diag["cedula"].map(cedula_to_idx)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Diagnóstico de errores — {best_name} LOAO-CV N2', fontsize=13, fontweight='bold')

# 1. MAE por atleta (barras con n_sesiones como opacidad)
ax = axes[0, 0]
mae_per_ath = df_diag.groupby("athlete_id")["abs_error_sec"].mean().sort_values()
colors_ath = [CRIMSON if v > mae_per_ath.median() else NAVY for v in mae_per_ath]
bars = ax.barh(mae_per_ath.index.astype(str), mae_per_ath.values, color=colors_ath, alpha=0.8)
ax.axvline(results_benchmark[best_name]["mae"], color='black', ls='--', lw=1.5,
           label=f'MAE global={results_benchmark[best_name]["mae"]:.0f} sec/km')
ax.set_xlabel('MAE (sec/km)')
ax.set_title('MAE por atleta\nRojo = sobre el promedio')
ax.legend(fontsize=9)

# 2. Distribución de errores (sesgo y varianza)
ax = axes[0, 1]
ax.hist(df_diag["error_sec"], bins=40, color=NAVY, alpha=0.8, edgecolor='white')
ax.axvline(0, color='black', lw=1.5)
ax.axvline(df_diag["error_sec"].mean(), color=CRIMSON, lw=2, ls='--',
           label=f'Media={df_diag["error_sec"].mean():.1f} sec/km')
ax.axvline(df_diag["error_sec"].median(), color='orange', lw=2, ls='--',
           label=f'Mediana={df_diag["error_sec"].median():.1f} sec/km')
ax.set_xlabel('Error predicción (sec/km)\n+ = predicción más lenta que real')
ax.set_ylabel('N sesiones')
ax.set_title('Distribución de errores\n(sesgo y simetría)')
ax.legend(fontsize=9)

# 3. MAE por género
ax = axes[1, 0]
mae_by_sex = df_diag.groupby("sex")["abs_error_sec"].agg(["mean", "count"])
if len(mae_by_sex) > 0:
    sex_labels = mae_by_sex.index.tolist()
    mae_vals = mae_by_sex["mean"].values
    ax.bar(sex_labels, mae_vals, color=[CRIMSON, NAVY][:len(sex_labels)], alpha=0.8)
    for i, (label, row) in enumerate(mae_by_sex.iterrows()):
        ax.text(i, row["mean"] + 0.5, f'n={int(row["count"])}', ha='center', fontsize=10)
    ax.set_ylabel('MAE medio (sec/km)')
    ax.set_title('MAE por género')
    ax.axhline(results_benchmark[best_name]["mae"], color='gray', ls='--', lw=1.5,
               label='MAE global')
    ax.legend(fontsize=9)

# 4. MAE por zona HR
ax = axes[1, 1]
df_diag["zona_hr_label"] = df_diag["zona_hr_w"].map(
    {1: "Z1 (<60%)", 2: "Z2 (60-70%)", 3: "Z3 (70-80%)", 4: "Z4 (80-90%)", 5: "Z5 (≥90%)"}
)
mae_by_zone = (df_diag[df_diag["zona_hr_w"].notna()]
               .groupby("zona_hr_label")["abs_error_sec"]
               .agg(["mean", "count"])
               .sort_index())
if len(mae_by_zone) > 0:
    zone_colors = ['#93C5FD', '#6EE7B7', '#FCD34D', '#F87171', '#C41E3A']
    bars_z = ax.bar(range(len(mae_by_zone)), mae_by_zone["mean"].values,
                    color=zone_colors[:len(mae_by_zone)], alpha=0.85)
    ax.set_xticks(range(len(mae_by_zone)))
    ax.set_xticklabels(mae_by_zone.index, rotation=20, ha='right')
    for i, (idx, row) in enumerate(mae_by_zone.iterrows()):
        ax.text(i, row["mean"] + 0.3, f'n={int(row["count"])}', ha='center', fontsize=9)
    ax.set_ylabel('MAE medio (sec/km)')
    ax.set_title('MAE por zona de intensidad HR')
    ax.axhline(results_benchmark[best_name]["mae"], color='gray', ls='--', lw=1.5,
               label='MAE global')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig_diagnostico.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumen estadístico
print("\nRESUMEN DE ERRORES")
print(f"  Sesgo medio (bias):   {df_diag['error_sec'].mean():+.1f} sec/km")
print(f"  Mediana error:        {df_diag['error_sec'].median():+.1f} sec/km")
print(f"  Desv. estándar:       {df_diag['error_sec'].std():.1f} sec/km")
print(f"  P10 / P90:            {df_diag['error_sec'].quantile(0.1):.0f} / {df_diag['error_sec'].quantile(0.9):.0f} sec/km")
if abs(df_diag['error_sec'].mean()) < 5:
    print("  → ✅ Modelo sin sesgo sistemático significativo")
else:
    dir_bias = "subestima el ritmo (predice más rápido)" if df_diag['error_sec'].mean() < 0 else "sobreestima el ritmo (predice más lento)"
    print(f"  → ⚠️  Sesgo detectado: {dir_bias}")

## B · 7 — Calibración conformal para N2

Split conformal: 80% atletas entrenamiento → reentrenar ganador → 20% calibración (errores reales) → cuantil q → test final.

Mismo protocolo α=0.20 (cobertura nominal 80%) que N1.

In [ ]:
ALPHA_CONF = 0.20  # → cobertura nominal 80%

# Partición por atletas (agrupada), no por sesiones
athletes_unique = df_model_n2["cedula"].unique()
np.random.seed(42)
np.random.shuffle(athletes_unique)

n_total = len(athletes_unique)
n_train = int(n_total * 0.60)
n_cal   = int(n_total * 0.20)

athletes_train = set(athletes_unique[:n_train])
athletes_cal   = set(athletes_unique[n_train:n_train + n_cal])
athletes_test  = set(athletes_unique[n_train + n_cal:])

mask_train = df_model_n2["cedula"].isin(athletes_train)
mask_cal   = df_model_n2["cedula"].isin(athletes_cal)
mask_test  = df_model_n2["cedula"].isin(athletes_test)

X_tr_c = X_n2[mask_train]; y_tr_c = y_n2[mask_train]; w_tr_c = sample_weights_n2[mask_train]
X_cal_c = X_n2[mask_cal];  y_cal_c = y_n2[mask_cal]
X_te_c  = X_n2[mask_test]; y_te_c  = y_n2[mask_test]

print(f"Split conformal por atleta:")
print(f"  Train:  {sum(mask_train)} sesiones / {len(athletes_train)} atletas")
print(f"  Cal:    {sum(mask_cal)} sesiones / {len(athletes_cal)} atletas")
print(f"  Test:   {sum(mask_test)} sesiones / {len(athletes_test)} atletas")

# Reentrenar ganador en train
from sklearn.linear_model import Ridge  # winner fallback si aún no se sabe
win_fn = {"Ridge": lambda: Ridge(alpha=1.0),
          "Lasso": lambda: Lasso(alpha=0.01),
          "ElasticNet": lambda: ElasticNet(alpha=0.01, l1_ratio=0.5),
          "GradBoost": lambda: GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=0),
          "HistGB": lambda: HistGradientBoostingRegressor(max_iter=200, random_state=0),
          "MLP": lambda: MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500, random_state=0),
          "RandomForest": lambda: RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1),
         }.get(best_name, lambda: Ridge(alpha=1.0))

scaler_c = StandardScaler().fit(X_tr_c)
model_c  = win_fn()
try:
    model_c.fit(scaler_c.transform(X_tr_c), y_tr_c, sample_weight=w_tr_c)
except TypeError:
    model_c.fit(scaler_c.transform(X_tr_c), y_tr_c)

# Errores de calibración
preds_cal = model_c.predict(scaler_c.transform(X_cal_c))
cal_errors = np.abs(y_cal_c - preds_cal)

# Cuantil conformal
q = np.quantile(cal_errors, 1 - ALPHA_CONF)
print(f"\nCalibración conformal (α={ALPHA_CONF}, nominal {(1-ALPHA_CONF)*100:.0f}%):")
print(f"  n_cal = {len(y_cal_c)} sesiones")
print(f"  Cuantil q = ±{q*60:.1f} sec/km  (±{q:.3f} min/km)")

# Verificar cobertura en test
preds_test = model_c.predict(scaler_c.transform(X_te_c))
lower = preds_test - q
upper = preds_test + q
covered = ((y_te_c >= lower) & (y_te_c <= upper)).mean()
print(f"\nCobertura empírica en test:")
print(f"  {covered*100:.1f}% (nominal {(1-ALPHA_CONF)*100:.0f}%)")
print(f"  Diferencia: {(covered - (1-ALPHA_CONF))*100:+.1f} pp")
if abs(covered - (1-ALPHA_CONF)) <= 0.05:
    print("  → ✅ Calibración correcta (diferencia ≤ 5 pp)")
else:
    print("  → ⚠️  Diferencia notable — revisar con más atletas en calibración")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Calibración conformal N2 — α={ALPHA_CONF} (nominal {(1-ALPHA_CONF)*100:.0f}%)', fontsize=12, fontweight='bold')

ax = axes[0]
ax.hist(cal_errors * 60, bins=30, color=NAVY, alpha=0.8, edgecolor='white')
ax.axvline(q * 60, color=CRIMSON, lw=2, ls='--', label=f'q={q*60:.0f} sec/km (p{int((1-ALPHA_CONF)*100)})')
ax.set_xlabel('|Error de calibración| (sec/km)')
ax.set_ylabel('N sesiones de calibración')
ax.set_title('Distribución de errores de calibración')
ax.legend(fontsize=10)

ax = axes[1]
n_show_conf = min(200, len(y_te_c))
idx_show = np.random.choice(len(y_te_c), n_show_conf, replace=False)
ax.scatter(y_te_c[idx_show], preds_test[idx_show], alpha=0.4, s=20, color=NAVY, zorder=5, label='Predicción')
ax.errorbar(y_te_c[idx_show], preds_test[idx_show],
            yerr=q, fmt='none', alpha=0.1, color=CRIMSON, zorder=3)
mn = min(y_te_c.min(), preds_test.min()); mx = max(y_te_c.max(), preds_test.max())
ax.plot([mn, mx], [mn, mx], 'k--', lw=1.5, label='Perfecto')
ax.set_xlabel('Ritmo real (min/km)')
ax.set_ylabel('Ritmo predicho ± q')
ax.set_title(f'Test set: cobertura={covered*100:.1f}% (muestra {n_show_conf} sesiones)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig_conformal.png', dpi=150, bbox_inches='tight')
plt.show()

## B · 8 — Serializar artefacto N2 y resumen de resultados canónicos

In [ ]:
import datetime

# ── Reentrenar modelo ganador en TODOS los datos para el artefacto ────────
scaler_final = StandardScaler().fit(X_n2)
model_final  = win_fn()
try:
    model_final.fit(scaler_final.transform(X_n2), y_n2, sample_weight=sample_weights_n2)
except TypeError:
    model_final.fit(scaler_final.transform(X_n2), y_n2)

# ── Artefacto ─────────────────────────────────────────────────────────────
artefacto_n2 = {
    "model": model_final,
    "scaler": scaler_final,
    "features": FEATURES_N2,
    "target": "pace_min_km",
    "model_name": best_name,
    "architecture": "cohort-level generalizable (N2)",
    "n_athletes": int(df_model_n2["cedula"].nunique()),
    "n_sessions": int(len(X_n2)),
    "loao_cv": {
        name: {"mae_sec_km": float(res["mae"]), "r2": float(res["r2"])}
        for name, res in results_benchmark.items()
    },
    "conformal": {
        "alpha": float(ALPHA_CONF),
        "nominal_coverage": float(1 - ALPHA_CONF),
        "q_min_km": float(q),
        "q_sec_km": float(q * 60),
        "empirical_coverage": float(covered),
        "n_cal": int(len(y_cal_c)),
    },
    "friedman": {
        "chi2": float(friedman_stat),
        "p_value": float(friedman_p),
        "n_folds": int(n_folds_common),
    },
    "n1_stacking": n1_features_n1,
    "sample_weighting": "w = 1/n_sesiones_atleta",
    "exclusions": ["CTL/ATL/TSB/ACWR → N3", "log_duration → circular", "PRs declarados → sesgo"],
    "date": datetime.date.today().isoformat(),
    "version": "2.0-nb13-corrected",
    "dataset_source": "RUNA/Strava — consentimiento explícito (Ley 1581/2012)",
}

out_pkl_n2 = OUT_DIR / f'nivel2_runa_v2_{best_name.lower().replace(" ", "_")}.pkl'
with open(out_pkl_n2, 'wb') as f:
    pickle.dump(artefacto_n2, f)
print(f"Artefacto serializado: {out_pkl_n2}")

# ── Resumen JSON para THESIS_CONTEXT.md ──────────────────────────────────
resumen_final = {
    "notebook": "NB13",
    "version": "2.0-nb13-corrected",
    "fecha": datetime.date.today().isoformat(),
    "arquitectura_n2": "cohort-level generalizable (SIN CTL/ATL/ACWR)",
    "n_atletas_total": int(df_model_n2["cedula"].nunique()),
    "n_sesiones_total": int(len(X_n2)),
    "n_features": len(FEATURES_N2),
    "features": FEATURES_N2,
    "loao_cv_ranking": [
        {"model": name, "mae_sec_km": round(res["mae"], 2), "r2": round(res["r2"], 3)}
        for name, res in sorted(results_benchmark.items(), key=lambda x: x[1]["mae"])
    ],
    "baseline_n1_puro": {"mae_sec_km": round(mae_n1_pure, 2)},
    "ganador": best_name,
    "friedman": {"chi2": round(friedman_stat, 2), "p_value": round(friedman_p, 4), "n_folds": n_folds_common},
    "conformal": {
        "alpha": ALPHA_CONF,
        "q_sec_km": round(q * 60, 1),
        "empirical_coverage": round(covered, 4),
        "n_cal": int(len(y_cal_c)),
    },
}

out_json = OUT_DIR / 'nb13_resultados_v2.json'
with open(out_json, 'w', encoding='utf-8') as f:
    import json
    json.dump(resumen_final, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 65)
print("RESUMEN CANÓNICO NB13 — Nivel 2 RUNA (versión 2026-05-12)")
print("=" * 65)
print(json.dumps(resumen_final, indent=2, ensure_ascii=False))

print("\n" + "=" * 65)
print("PRÓXIMOS PASOS → NB14 — Nivel 3 longitudinal individual")
print("=" * 65)
print("""
NB14 diseñará el Nivel 3 con:
  - Features adicionales: CTL, ATL, TSB, ACWR (de weekly_features Supabase)
  - own_race_best_pace, n_races_registered, weeks_available
  - Activación: ≥3 carreras propias (sin check-ins)
  - pred_nivel2 como stacking feature (Wolpert 1992 — chain jerárquica)
  - Manejo de missingness: imputa por mediana del fold de entrenamiento
  - LOAO-CV extendido (mismos 32 folds)
  - Pregunta: ¿cuándo N3 supera N2 estadísticamente? → curva de ganancia

Umbral para ejecutar NB14:
  - ≥10 atletas con ≥3 carreras propias registradas en Supabase
  - ≥8 semanas de CTL/ATL en weekly_features

Estado actual: por verificar en Supabase (n_races por atleta).
""")